# Ablation Study Evaluation

Evaluates all ablation runs on the test split and compares against the fair baseline and full improved model.

**Experiments:**
- `net_fish_sonar` — YOLOv26s sonar detector: optimizer · LR schedule · mosaic · geometric aug · random erasing
- `solaqua_fish` — RT-DETR-L vision detector: LR schedule · HSV aug · geometric aug · random erasing · mosaic (negative test)

Cells skip any run whose `best.pt` is not yet present — re-run after training completes.

In [ ]:
import sys
sys.path.insert(0, "/workspace/aquaculture-perception")

import os
from pathlib import Path
from typing import Optional
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from ultralytics import YOLO
from utils.run_registry import load_runs_csv
from utils.paths import objdet_root, resolve_from_objdet

root = objdet_root()
runs = load_runs_csv(root / "runs.csv")
os.makedirs("outputs/evaluation/ablation", exist_ok=True)
print(f"Loaded {len(runs)} runs")
print(f"Root: {root}")

In [ ]:
def find_weights(run_id: str) -> Path:
    """Try both current-project and legacy runs/detect output locations."""
    run = runs[run_id]
    project = run.get("project", "")
    name = run.get("name", run_id)
    candidates = [
        root / project / name / "weights" / "best.pt",
        root.parent / "runs" / "detect" / project / name / "weights" / "best.pt",
    ]
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]


def evaluate_run(run_id: str, data_yaml: str, label: str) -> Optional[dict]:
    weights = find_weights(run_id)
    if not weights.exists():
        print(f"  [SKIP] {run_id} — weights not found at {weights}")
        return None
    print(f"  Evaluating: {run_id}")
    model = YOLO(str(weights))
    results = model.val(
        data=data_yaml,
        split="test",
        imgsz=640,
        batch=1,
        iou=0.45,
        conf=0.25,
        plots=False,
        project="outputs/evaluation/ablation",
        name=f"EVAL_{run_id}",
        exist_ok=True,
        verbose=False,
    )
    b = results.box
    return {
        "run_id": run_id,
        "label": label,
        "mAP50": round(b.map50, 4),
        "mAP50-95": round(b.map, 4),
        "Precision": round(b.mp, 4),
        "Recall": round(b.mr, 4),
    }


COLORS = {"baseline": "#aaaaaa", "improved": "#2ecc71", "ablation": "#3498db"}


def ablation_barplot(
    df: pd.DataFrame,
    run_configs: list,
    metric: str = "mAP50",
    title: str = "",
    save_path: Optional[str] = None,
):
    entries = [(lbl, kind) for _, lbl, kind in run_configs if lbl in df["label"].values]
    labels = [e[0] for e in entries]
    colors = [COLORS[e[1]] for e in entries]
    values = [df.loc[df["label"] == lbl, metric].values[0] for lbl in labels]

    fig, ax = plt.subplots(figsize=(9, max(3.5, len(labels) * 0.65)))
    bars = ax.barh(labels, values, color=colors, edgecolor="white", height=0.6)

    x_max = min(1.0, max(values) * 1.18)
    for bar, val in zip(bars, values):
        ax.text(
            val + x_max * 0.005,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}",
            va="center", ha="left", fontsize=9,
        )

    improved_rows = df[df["label"] == "Improved (full)"]
    if not improved_rows.empty:
        ref = improved_rows[metric].values[0]
        ax.axvline(ref, color="#27ae60", linestyle="--", linewidth=1.2, alpha=0.8)

    ax.set_xlabel(metric, fontsize=11)
    ax.set_title(title, fontsize=12, pad=10)
    ax.invert_yaxis()
    ax.set_xlim(0, x_max)
    ax.grid(axis="x", linestyle=":", alpha=0.4)

    patches = [
        mpatches.Patch(color=COLORS["baseline"], label="Fair baseline"),
        mpatches.Patch(color=COLORS["improved"], label="Improved (full)"),
        mpatches.Patch(color=COLORS["ablation"], label="Ablation (remove one component)"),
    ]
    ax.legend(handles=patches, fontsize=8, loc="lower right")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    return fig


def results_table(rows: list, ref_label: str = "Improved (full)") -> pd.DataFrame:
    df = pd.DataFrame(rows)
    if ref_label in df["label"].values:
        ref_map50 = df.loc[df["label"] == ref_label, "mAP50"].values[0]
        ref_map5095 = df.loc[df["label"] == ref_label, "mAP50-95"].values[0]
        df["Delta mAP50"] = (df["mAP50"] - ref_map50).round(4)
        df["Delta mAP50-95"] = (df["mAP50-95"] - ref_map5095).round(4)
    return df

---
## 1. Net Fish Sonar — YOLOv26s Ablation

Dataset: 1 200 train / 400 val / 400 test · 2 classes: `fish`, `net` · sonar domain

| Run | What changes |
|---|---|
| Fair baseline | 120e, SGD, fliplr only |
| Improved (full) | AdamW, cosine LR, 300e, full aug |
| Abl: No cosine LR | Linear LR decay instead of cosine |
| Abl: No aug | All augmentation removed except fliplr |


In [ ]:
SONAR_DATA = str(resolve_from_objdet("../data-processing/sonar/net_fish_sonar/net_fish_sonar.yaml"))

SONAR_RUNS = [
    ("yolov26s_net_fish_sonar_120e_fair", "Fair baseline",    "baseline"),
    ("yolov26s_sonar_improved",           "Improved (full)",  "improved"),
    ("yolov26s_sonar_abl_no_cos",         "Abl: No cosine LR","ablation"),
    ("yolov26s_sonar_abl_no_aug",         "Abl: No aug",      "ablation"),
]

In [ ]:
print("=== Net Fish Sonar — YOLOv26s ===")
sonar_rows = []
for run_id, label, _ in SONAR_RUNS:
    row = evaluate_run(run_id, SONAR_DATA, label)
    if row:
        sonar_rows.append(row)
print(f"\nCompleted {len(sonar_rows)}/{len(SONAR_RUNS)} runs.")

In [ ]:
df_sonar = results_table(sonar_rows)

display_cols = ["label", "mAP50", "mAP50-95", "Precision", "Recall", "Delta mAP50", "Delta mAP50-95"]
display_cols = [c for c in display_cols if c in df_sonar.columns]

print("Net Fish Sonar — YOLOv26s Ablation Results")
print("=" * 75)
print(df_sonar[display_cols].to_string(index=False))

In [ ]:
if sonar_rows:
    ablation_barplot(
        df_sonar, SONAR_RUNS,
        metric="mAP50",
        title="Net Fish Sonar — YOLOv26s Ablation Study (mAP50)",
        save_path="outputs/evaluation/ablation/sonar_ablation_map50.png",
    )
    plt.show()

    ablation_barplot(
        df_sonar, SONAR_RUNS,
        metric="mAP50-95",
        title="Net Fish Sonar — YOLOv26s Ablation Study (mAP50-95)",
        save_path="outputs/evaluation/ablation/sonar_ablation_map5095.png",
    )
    plt.show()

---
## 2. Solaqua Fish — RT-DETR-L Ablation

Dataset: 505 train / 124 val / 164 test · 1 class: `fish` · vision domain

| Run | What changes |
|---|---|
| Fair baseline | 120e, SGD, fliplr only |
| Improved (full) | AdamW, cosine LR, 300e, full aug |
| Abl: No cosine LR | Linear LR decay instead of cosine |
| Abl: No aug | All augmentation removed except fliplr |
| Abl: +Mosaic | Tests if mosaic hurts RT-DETR (expected: yes) |

> RT-DETR disables mosaic by default — its bipartite Hungarian matching is disrupted
> by objects from 4 different images being stitched together (*Zhao & Lv et al., CVPR 2024*).

In [ ]:
FISH_DATA = str(resolve_from_objdet("../data-processing/vision/solaqua_fish/solaqua_fish.yaml"))

FISH_RUNS = [
    ("rt_detr_solaqua_fish_120e_fair", "Fair baseline",    "baseline"),
    ("rtdetr_fish_improved",           "Improved (full)",  "improved"),
    ("rtdetr_fish_abl_no_cos",         "Abl: No cosine LR","ablation"),
    ("rtdetr_fish_abl_no_aug",         "Abl: No aug",      "ablation"),
    ("rtdetr_fish_abl_mosaic",         "Abl: +Mosaic",     "ablation"),
]

In [ ]:
print("=== Solaqua Fish — RT-DETR-L ===")
fish_rows = []
for run_id, label, _ in FISH_RUNS:
    row = evaluate_run(run_id, FISH_DATA, label)
    if row:
        fish_rows.append(row)
print(f"\nCompleted {len(fish_rows)}/{len(FISH_RUNS)} runs.")

In [ ]:
df_fish = results_table(fish_rows)

display_cols = ["label", "mAP50", "mAP50-95", "Precision", "Recall", "Delta mAP50", "Delta mAP50-95"]
display_cols = [c for c in display_cols if c in df_fish.columns]

print("Solaqua Fish — RT-DETR-L Ablation Results")
print("=" * 75)
print(df_fish[display_cols].to_string(index=False))

In [ ]:
if fish_rows:
    ablation_barplot(
        df_fish, FISH_RUNS,
        metric="mAP50",
        title="Solaqua Fish — RT-DETR-L Ablation Study (mAP50)",
        save_path="outputs/evaluation/ablation/fish_ablation_map50.png",
    )
    plt.show()

    ablation_barplot(
        df_fish, FISH_RUNS,
        metric="mAP50-95",
        title="Solaqua Fish — RT-DETR-L Ablation Study (mAP50-95)",
        save_path="outputs/evaluation/ablation/fish_ablation_map5095.png",
    )
    plt.show()

In [ ]:
if sonar_rows:
    out = "outputs/evaluation/ablation/sonar_ablation_results.csv"
    df_sonar.to_csv(out, index=False)
    print(f"Saved: {out}")

if fish_rows:
    out = "outputs/evaluation/ablation/fish_ablation_results.csv"
    df_fish.to_csv(out, index=False)
    print(f"Saved: {out}")

---
## 3. Optimizer LR Sweep (Phase 1)

Run before the improved model. Finds the best optimizer and LR — those settings are then locked in for Phase 2.
All other settings are identical to the fair baseline (fliplr only, no cosine LR).

**YOLOv26s (sonar):** SGD and AdamW both tested.
| Optimizer | LR values |
|---|---|
| SGD | 0.01 · 0.005 · 0.001 |
| AdamW | 0.001 · 0.0005 · 0.0001 |

**RT-DETR-L (vision):** AdamW only — transformer decoders require adaptive optimizers (*Carion et al., ECCV 2020*).
| Optimizer | LR values |
|---|---|
| AdamW | 0.001 · 0.0001 · 0.00001 |

In [ ]:
def evaluate_opt_run(run_id: str, data_yaml: str, optimizer: str, lr: float) -> Optional[dict]:
    label = f"{optimizer} lr={lr:.0e}"
    row = evaluate_run(run_id, data_yaml, label)
    if row:
        row["optimizer"] = optimizer
        row["lr"] = lr
    return row


def lr_sweep_plot(
    rows: list,
    metric: str = "mAP50",
    title: str = "",
    save_path: Optional[str] = None,
):
    df = pd.DataFrame(rows)
    if df.empty:
        print("No data to plot.")
        return

    opt_colors = {"SGD": "#e74c3c", "AdamW": "#3498db"}
    fig, ax = plt.subplots(figsize=(7, 4))

    for opt, grp in df.groupby("optimizer"):
        grp = grp.sort_values("lr")
        color = opt_colors.get(opt, "#555")
        ax.plot(grp["lr"], grp[metric], marker="o", label=opt,
                color=color, linewidth=2, markersize=7)
        for _, row in grp.iterrows():
            ax.annotate(
                f"{row[metric]:.4f}",
                (row["lr"], row[metric]),
                textcoords="offset points", xytext=(0, 9),
                ha="center", fontsize=8, color=color,
            )

    ax.set_xscale("log")
    ax.set_xlabel("Learning Rate (log scale)", fontsize=11)
    ax.set_ylabel(metric, fontsize=11)
    ax.set_title(title, fontsize=12, pad=10)
    ax.legend(fontsize=10)
    ax.grid(True, linestyle=":", alpha=0.4)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    return fig

### 3a. Net Fish Sonar — YOLOv26s Optimizer Sweep

In [ ]:
# (run_id, optimizer, lr)
SONAR_OPT_RUNS = [
    ("yolov26s_sonar_opt_sgd_lr01",     "SGD",   0.01),
    ("yolov26s_sonar_opt_sgd_lr005",    "SGD",   0.005),
    ("yolov26s_sonar_opt_sgd_lr001",    "SGD",   0.001),
    ("yolov26s_sonar_opt_adamw_lr001",  "AdamW", 0.001),
    ("yolov26s_sonar_opt_adamw_lr0005", "AdamW", 0.0005),
    ("yolov26s_sonar_opt_adamw_lr0001", "AdamW", 0.0001),
]

In [ ]:
print("=== Net Fish Sonar — Optimizer LR Sweep ===")
sonar_opt_rows = []
for run_id, optimizer, lr in SONAR_OPT_RUNS:
    row = evaluate_opt_run(run_id, SONAR_DATA, optimizer, lr)
    if row:
        sonar_opt_rows.append(row)
print(f"\nCompleted {len(sonar_opt_rows)}/{len(SONAR_OPT_RUNS)} runs.")

In [ ]:
if sonar_opt_rows:
    df_sonar_opt = pd.DataFrame(sonar_opt_rows)
    display_cols = ["optimizer", "lr", "mAP50", "mAP50-95", "Precision", "Recall"]
    display_cols = [c for c in display_cols if c in df_sonar_opt.columns]
    print("Net Fish Sonar — Optimizer LR Sweep")
    print("=" * 60)
    print(df_sonar_opt[display_cols].sort_values(["optimizer", "lr"]).to_string(index=False))
    print()

    best_sgd   = df_sonar_opt[df_sonar_opt["optimizer"] == "SGD"].sort_values("mAP50", ascending=False)
    best_adamw = df_sonar_opt[df_sonar_opt["optimizer"] == "AdamW"].sort_values("mAP50", ascending=False)
    if not best_sgd.empty:
        r = best_sgd.iloc[0]
        print(f"Best SGD:   lr={r['lr']:.0e}  mAP50={r['mAP50']:.4f}")
    if not best_adamw.empty:
        r = best_adamw.iloc[0]
        print(f"Best AdamW: lr={r['lr']:.0e}  mAP50={r['mAP50']:.4f}")

In [ ]:
if sonar_opt_rows:
    lr_sweep_plot(
        sonar_opt_rows,
        metric="mAP50",
        title="Net Fish Sonar — YOLOv26s Optimizer LR Sweep (mAP50)",
        save_path="outputs/evaluation/ablation/sonar_opt_lr_sweep.png",
    )
    plt.show()

### 3b. Solaqua Fish — RT-DETR-L Optimizer Sweep

> SGD is not tested for RT-DETR: the transformer decoder relies on per-parameter adaptive gradients and diverges under SGD (*Lv et al., CVPR 2024*; *DN-DETR, CVPR 2022*).

In [ ]:
FISH_OPT_RUNS = [
    ("rtdetr_fish_opt_adamw_lr001",   "AdamW", 0.001),
    ("rtdetr_fish_opt_adamw_lr0001",  "AdamW", 0.0001),
    ("rtdetr_fish_opt_adamw_lr00001", "AdamW", 0.00001),
]

In [ ]:
print("=== Solaqua Fish — RT-DETR Optimizer LR Sweep ===")
fish_opt_rows = []
for run_id, optimizer, lr in FISH_OPT_RUNS:
    row = evaluate_opt_run(run_id, FISH_DATA, optimizer, lr)
    if row:
        fish_opt_rows.append(row)
print(f"\nCompleted {len(fish_opt_rows)}/{len(FISH_OPT_RUNS)} runs.")

In [ ]:
if fish_opt_rows:
    df_fish_opt = pd.DataFrame(fish_opt_rows)
    display_cols = ["optimizer", "lr", "mAP50", "mAP50-95", "Precision", "Recall"]
    display_cols = [c for c in display_cols if c in df_fish_opt.columns]
    print("Solaqua Fish — RT-DETR-L AdamW LR Sweep")
    print("=" * 60)
    print(df_fish_opt[display_cols].sort_values("lr").to_string(index=False))
    print()

    best = df_fish_opt.sort_values("mAP50", ascending=False).iloc[0]
    print(f"Best AdamW: lr={best['lr']:.0e}  mAP50={best['mAP50']:.4f}")

In [ ]:
if fish_opt_rows:
    lr_sweep_plot(
        fish_opt_rows,
        metric="mAP50",
        title="Solaqua Fish — RT-DETR-L AdamW LR Sweep (mAP50)",
        save_path="outputs/evaluation/ablation/fish_opt_lr_sweep.png",
    )
    plt.show()

In [ ]:
if sonar_opt_rows:
    out = "outputs/evaluation/ablation/sonar_opt_sweep_results.csv"
    pd.DataFrame(sonar_opt_rows).to_csv(out, index=False)
    print(f"Saved: {out}")

if fish_opt_rows:
    out = "outputs/evaluation/ablation/fish_opt_sweep_results.csv"
    pd.DataFrame(fish_opt_rows).to_csv(out, index=False)
    print(f"Saved: {out}")